# Promotion: research to paper to live

A strategy moves between stages as a **spec** (`research/strategies/<id>.toml`): the same file research backtests is
what the paper and live books run (`strategy = "lab"` in a portfolio sleeve, pinned to a version). This notebook
gathers the evidence, runs the gates, writes the factsheet, and records the decision.

| gate | paper | live |
| --- | --- | --- |
| strategy card | not "no evidence" | "evidence of an edge" |
| power, holdout look | reported | required |
| costs (Sharpe > 0 at 2x), capacity, risk budget, owner sign-off | required | required |

A failing gate can be overridden only with a written reason, which stays with the promotion. The owner's sign-off
cannot be overridden.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantlab.lab import Lab
from quantlab.signals import library as lib
from quantlab.reporting import charts

charts.apply_style()
pd.set_option("display.width", 200)

from IPython.display import HTML, display
from quantlab.deploy.spec import load_spec, find_spec
from quantlab.deploy import gates
from quantlab.reporting.factsheet import build_factsheet

spec = load_spec(find_spec("smooth_trend"))
print(spec.key, spec.hash)
print(spec.strategy.description)

## Evidence

Backtest, strategy card (noted under the spec's hypothesis), power, costs at twice the assumption, capacity by capital.

In [ ]:
evidence = gates.evaluate(spec, start="2005-07-01")
print(f"{evidence.years:.1f} years; card: {evidence.card.verdict} ({evidence.card.reason})")
evidence.capacity["table"].round(4)

## The gates

Set the budget the book can give this strategy (capital in USD, the volatility and drawdown it may use).

In [ ]:
paper_budget = gates.RiskBudget(capital_usd=1e6, max_volatility=0.15, max_drawdown=0.30)
paper = gates.promotion_check(spec, "paper", evidence=evidence, budget=paper_budget)
print(paper)
paper.table

In [ ]:
live = gates.promotion_check(spec, "live", evidence=evidence, budget=gates.RiskBudget(capital_usd=1e8, max_drawdown=0.20))
print(live)
live.table

## Factsheet

In [ ]:
path = build_factsheet(spec, evidence, gates=live, correlations=None, stage="paper (H13 book, not yet promoted through the gates)")
display(HTML(f'<a href="{path.resolve().as_uri()}" target="_blank">open the factsheet</a>'))

## The decision (the owner runs these; nothing below runs by itself)

```python
gates.sign_off(spec, "paper", by="owner", note="why")
gates.promote(spec, "paper", evidence=evidence, approved_by="owner", budget=paper_budget,
              overrides={"cards": "H13 is already on paper to collect live evidence; the card says no evidence"})
```

Then a sleeve may run it, and `python main.py --portfolio-check <book>.toml --require-strategy-cards` passes:

```toml
[[sleeves]]
strategy = "lab"
params = { spec = "smooth_trend", version = 1 }
continuous = true
```

Once the book has run for a while: `python scripts/lab/parity_check.py --config <book>.toml --spec smooth_trend`
compares it with the research engine and with the spec's expectations.